# prototype clock value in jax
Make a working clock-value code with jax and jax-finufft that is fast enough!

## author:
- **David W. Hogg** (NYU) (Flatiron) (MPIA)

## bugs:
- The removal of resonant clocks involves a lot of MAGIC that might need to be revisited.
- The `best_clocks_in_star()` function should make more plots than just the final one.
- Maybe, when a light curve contains multiple clocks, the clocks should be fit simultaneously and the empirical clock value computed in that context?

## comments:
- Because my Mac has issues with jax, only use jax when we absolutely need it.

In [ ]:
# !pip install lightkurve
# !pip install jax

In [ ]:
from functools import partial
from fractions import Fraction
import numpy as np
import jax
import jax.numpy as jnp
import lightkurve as lk
from astropy.timeseries import LombScargle
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
import time

In [ ]:
jax.config.update('jax_platform_name', 'cpu') # because bad Mac behavior?
jax.config.update("jax_enable_x64", True)

In [ ]:
# set high-level parameters of the method
MAX_PERIOD = 30. # days
MIN_VALUE = 1.e7 # inverse days squared
MIN_OPTIMISTIC_VALUE = 1.e9 # inverse days squared

In [ ]:
def get_kepler_data(kic_id, exptime='long'):
    """
    ## Inputs:
    `kic_id`: Kepler ID (str)
    `exptime`: default exposure time, 'long'

    ## Outputs:
    Returns a 5-tuple:
    - `times`, `fluxes`, `errors`: light curve data
    - `delta_f`: frequency resolution, 1 / total observation time
    - `sampling_time`: median time between observations (in days)
    """    
    start = time.time()
    print("starting to obtain data for", kic_id)

    try:
        search_result = lk.search_lightcurve(kic_id, mission = 'Kepler', exptime=exptime)
        if len(search_result) < 1:
            msg = f"get_kepler_data(): no results for {kic_id} at this cadence"
            print(msg)
            update_error_message(kic_id, 'Kepler_long', msg)
            return None, None, None, None, None
    except Exception as e:
        print(f"Exception for {kic_id}: get_kepler_data(): lk.search_lightcurve() failed for {kic_id } with {str(e)}")
        return None, None, None, None, None

    try:
        lc_collection = search_result.download_all()
    except lk.LightkurveError as e:
        print(f"LightkurveError for {kic_id}: get_kepler_data(): search_result.download_all() failed for {kic_id} with {str(e)}")
        return None, None, None, None, None
    except Exception as e:
        print(f"Exception for {kic_id}: get_kepler_data(): search_result.download_all() failed for {kic_id} with {str(e)}")
        return None, None, None, None, None

    try:
        lc = lc_collection.stitch()
        #print("get_kepler_data(): minimum time value", np.min(lc.time.value), np.min(lc.time), lc.time)
    except lk.LightkurveError as e:
        print(f"LightkurveError for {kic_id}: get_kepler_data(): lc_collection.stitch() failed for {kic_id} with {str(e)}")
        return None, None, None, None, None
    except Exception as e:
        print(f"Exception for {kic_id}: get_kepler_data(): lc_collection.stitch() failed for {kic_id} with {str(e)}")
        return None, None, None, None, None

    # unpack, remove bad data, and reorder
    times, fluxes, errors = lc.time.value, lc.flux.value, lc.flux_err.value
    good = np.isfinite(times) & np.isfinite(fluxes) & np.isfinite(errors)
    times, fluxes, errors = times[good], fluxes[good], errors[good]
    idx = np.argsort(times)
    times, fluxes, errors = times[idx], fluxes[idx], errors[idx]

    delta_f = (1/(times[-1] - times[0]))
    sampling_time= np.median(np.diff(times))
    print("get_kepler_data() took", time.time() - start, "seconds")

    return times, fluxes, errors, delta_f, sampling_time

In [ ]:
# get a set of hypotheses for this star

def get_candidate_frequencies(ts, ys, errs, df, dt, max_peaks=32, nterms=8):
    """
    # inputs:
    - `ts`, `ys`, `errs`: the light curve
    - `df`, `dt`: the smallest frequency and the smallest time of relevance

    # bugs:
    - MAGIC oversampling by a factor of either 2 or 4 (I don't know which)
    """
    fs = np.arange(1. / MAX_PERIOD, 0.5 / dt, 0.25 * df)
    ps = LombScargle(ts, ys, errs, nterms=nterms).power(fs)
    idxs, _ = find_peaks(ps, distance=4)
    ii = np.argsort(ps[idxs])[::-1]
    idxs = idxs[ii]
    if len(idxs) > max_peaks:
        idxs = idxs[:max_peaks]
    return fs[idxs]

In [ ]:
# design matrix for an M-component Fourier series
# Note: solving this problem with standard (non-finufft) methods

@partial(jax.jit, static_argnums=2)
def design_matrix(om, t, M):
    ms1, ms2 = jnp.arange(M + 1), jnp.arange(1, M + 1)
    return jnp.concat((jnp.cos(ms1[None, :] * om * t[:, None]),
                       jnp.sin(ms2[None, :] * om * t[:, None])), axis=1), \
           jnp.concat((ms1, ms2))                    

In [ ]:
# time to get the clock value

@partial(jax.jit, static_argnums=4)
def fourier_wls_fit(om, t, y, iv, M):
    X, m = design_matrix(om, t, M)
    return X, m, jnp.linalg.solve(X.T @ (iv[:, None] * X),
                                  X.T @ (iv * y))

@partial(jax.jit, static_argnums=4)
def clock_value(om, t, y, iv, M):
    """
    # inputs:
    - `om`: frequency to test
    - `t`, `y`, `iv`: light curve (iv is inverse variance, not error)
    - `M`: degree of Fourier series

    # bugs:
    - This is very affected by outliers; need to remove those somehow. But *don't* use `jnp.median()`!
    - Maybe we should use IRLS to do the fit.
    - Maybe we should penalize (or increase) MSE according to model complexity `2 * M + 1`.
    """
    X, m, pars = fourier_wls_fit(om, t, y, iv, M)
    mse = jnp.sum(iv * (y - X @ pars) ** 2) / jnp.sum(iv) # weighted mean
    return len(y) * (om ** 2 / mse) * jnp.sum(m ** 2 * pars ** 2)

clock_values = jax.vmap(clock_value, in_axes=(0, None, None, None, None))

In [ ]:
@partial(jax.jit, static_argnums=4)
def optimistic_clock_value(om, t, y, iv, M):
    """
    I feel pretty confident about the 0.5 in there.
    """
    _, m, pars = fourier_wls_fit(om, t, y, iv, M)
    return 0.5 * np.sum(iv) * om ** 2 * jnp.sum(m ** 2 * pars ** 2)

def take_derivative_wrt_phase(ps, ms):
    M = (len(ms) - 1) // 2
    newps = np.zeros_like(ps)
    newps[1 : M + 1] = ms[M + 1 :] * ps[M + 1 :]
    newps[M + 1 :] = -1. * ms[1 : M + 1] * ps[1 : M + 1]
    return newps

def theoretical_clock_value(om, t, y, iv, M):
    X, m, pars = fourier_wls_fit(om, t, y, iv, M)
    dpars = take_derivative_wrt_phase(pars, m)
    derivs = X @ dpars
    return om ** 2 * np.sum(iv * derivs ** 2)

In [ ]:
# get best clock-value frequency near a peak
# Note: this is the parabola trick in the log

def get_best_clock(om0, t, y, iv, Mmax, df, dt):
    """
    # inputs:
    - `om0`: first guess at a good clock (angular) frequency omega
    - `t`, `y`, `iv`: the light curve
    - `Mmax`: the maximum degree of the Fourier series (not necessarily the degree)
    - `df`: the frequency resolution (non-angular frequency) in the data
    - `dt`: the sampling time (related to Nyquist).

    # bugs:
    - Takes one input as angular frequency and another as frequency.
    - MAGIC 0.05

    # notes:
    - Recursion is insane.
    - Works in the log for stability.
    """
    nyquist = np.pi / dt # angular-frequency units
    M = max(1, min(Mmax, int(nyquist // om0)))
    do = 0.05 * np.pi * df # magic 0.05
    oms = np.array([om0 - do, om0, om0 + do])
    ys = np.log(clock_values(oms, t, y, iv, M))
    if np.argmax(ys) != 1:
        return get_best_clock(oms[np.argmax(ys)], t, y, iv, M, df, dt)
    ii = jnp.argmax(ys)
    foo = jnp.polyfit(oms, ys, 2)
    om = jnp.roots(jnp.polyder(foo), strip_zeros=False).real
    return om[0], jnp.exp(jnp.polyval(foo, om))[0], M

In [ ]:
# clock plotting functions

def latex_sci_not(x):
    if x == 0:
        return "$0$"
    s = f"{x:.1e}"
    mantissa, exponent = s.split("e")
    return rf"${mantissa} \times 10^{{{int(exponent)}}}$"# look at best frequency

def set_plot_range(xs):
    a, b = np.percentile(xs, [0.025, 97.5])
    mid, dif = 0.5 * (b + a), 0.5 * (b - a)
    return mid - 2. * dif, mid + 2 * dif

def plot_clock(kicid, om, M, val, optval, theoval, ts, ys, ivs):
    """
    ## bugs
    - Note commented-out code that plots model derivative.
    """
    period = 2. * jnp.pi / om
    f = plt.figure(figsize=(9, 3))
    _, ms, pars = fourier_wls_fit(om, ts, ys, ivs, M)
    # dpars = take_derivative_wrt_phase(pars, ms)
    thetas_plot = np.linspace(0., 4. * np.pi, 1000)
    X_plot, _ = design_matrix(om, thetas_plot / om, M)
    plt.scatter((om * ts) % (2. * np.pi), ys, s=1, c="k", marker=".", alpha=0.5)
    plt.scatter((om * ts) % (2. * np.pi) + 2. * np.pi, ys, s=1, c="k", marker=".")
    plt.plot(thetas_plot, X_plot @ pars, "r-")
    # plt.plot(thetas_plot, X_plot @ dpars + pars[0], "r--")
    plt.title(f"{kicid}; period {period:.6f} d; value " + latex_sci_not(val)
              + r" d$^{-2}$ (emp) " + latex_sci_not(optval) + " (opt) " + latex_sci_not(theoval) + " (theo)")
    plt.xlim(0., 4. * np.pi)
    plt.ylim(set_plot_range(ys))
    plt.xlabel("phase [rad]")
    return f

In [ ]:
# put it all together into one huge function.

def identify_resonances(fs, tol=1.e-4, max_denominator=12):
    """
    # bug:
    - REQUIRES that the `fs` be ordered from highest value to lowest.
    - Lots of MAGIC.
    """
    duplicates = np.zeros_like(fs).astype(bool)
    for i, f in enumerate(fs):
        if not duplicates[i]:
            for j in range(i + 1, len(fs)):
                ratio_ji = fs[j] / fs[i]
                frac_ji = Fraction(ratio_ji).limit_denominator(max_denominator)
                test_ji = abs((ratio_ji - float(frac_ji)) / ratio_ji) < tol
                ratio_ij = fs[i] / fs[j]
                frac_ij = Fraction(ratio_ij).limit_denominator(max_denominator)
                test_ij = abs((ratio_ij - float(frac_ij)) / ratio_ij) < tol
                duplicates[j] = test_ji | test_ij
                print("identify_resonances():", fs[i], fs[j], ratio_ji, frac_ji, ratio_ij, frac_ij, duplicates[j])
    return duplicates

def best_clocks_in_star(kicid, Mmax=128, plot=True):
    print(f"best_clocks_in_star(): getting Kepler data for {kicid}")
    ts, ys, errs, deltaf, deltat = get_kepler_data(kicid)
    if ts is None:
        print(f"best_clocks_in_star(): skipping {kicid}")
        return [], [], [], []
    ivars = 1. / errs ** 2
    print(f"best_clocks_in_star(): getting candidate frequencies for {kicid}")
    candidate_oms = 2. * np.pi * get_candidate_frequencies(ts, ys, errs, deltaf, deltat)
    
    # now loop through candidates and refine them
    oms, values = np.zeros_like(candidate_oms), np.zeros_like(candidate_oms)
    Ms = np.zeros_like(candidate_oms).astype(int)
    print(f"best_clocks_in_star(): getting clock values for {kicid}")
    for i, om0 in enumerate(candidate_oms):
        oms[i], values[i], Ms[i] = get_best_clock(om0, ts, ys, ivars, Mmax, deltaf, deltat)
    optimistic_values = np.array([optimistic_clock_value(om, ts, ys, ivars, M) for om, M in zip(oms, Ms)])
    theoretical_values = np.array([theoretical_clock_value(om, ts, ys, ivars, M) for om, M in zip(oms, Ms)])
    
    # now filter and arrange the clocks
    good = (values > MIN_VALUE) | (optimistic_values > MIN_OPTIMISTIC_VALUE)
    if np.sum(good) < 1:
        return [], [], [], []
    oms, values, Ms, optimistic_values, theoretical_values = \
        oms[good], values[good], Ms[good], optimistic_values[good], theoretical_values[good]
    idx_sort = np.argsort(values)[::-1]
    oms, values, Ms, optimistic_values, theoretical_values = \
        oms[idx_sort], values[idx_sort], Ms[idx_sort], optimistic_values[idx_sort], theoretical_values[idx_sort]
    idx_remove = np.logical_not(identify_resonances(oms))
    oms, values, Ms, optimistic_values, theoretical_values = \
        oms[idx_remove], values[idx_remove], Ms[idx_remove], optimistic_values[idx_remove], theoretical_values[idx_remove]

    if plot:
        for om, value, optval, theoval, M in zip(oms, values, optimistic_values, theoretical_values, Ms):
            f = plot_clock(kicid, om, M, value, optval, theoval, ts, ys, ivars)
            plt.show()
    return oms, values, optimistic_values, Ms

In [ ]:
# now do a whole bunch of KICs

kics = [ "KIC002162994", "KIC005285607", "KIC003240411", "KIC003459297", "KIC003865742", "KIC004930889",
    "KIC004939281", "KIC005309849", "KIC005941844", "KIC006352430",
    "KIC006462033", "KIC006780397", "KIC007630417", "KIC007760680", "KIC008057661",
    "KIC008255796", "KIC008381949", "KIC008459899", "KIC008714886", "KIC008766405",
    "KIC009020774", "KIC009715425", "KIC010526294", "KIC011360704", "KIC011971405",
    "KIC012258330", "KIC000520290", "KIC009111849", "KIC009845898", "KIC006116172", "KIC011013201",
    "KIC005617259", "KIC009289704", "KIC007767699", "KIC008249829", "KIC010874614",
    "KIC009895543" ]
kics_fail =  [ "KIC005112738", "KIC005112705", "KIC005112855", "KIC012251995" ]
kics_boring = [ "KIC009041983", "KIC007022977", "KIC009224229" ]
kics_dumb =  [ "KIC007264961", "KIC009656543", "KIC009772694", "KIC010416004",
               "KIC004936089", ]
kics_pulse = [ "KIC006587551", "KIC003544595", "KIC011295426", "KIC009700322",
               "KIC006370665", "KIC002581626", "KIC009828226", "KIC008309815",
               "KIC008290073", "KIC007840896" ]
for kicid in kics:
    oms, vals, optvals, Ms = best_clocks_in_star(kicid)